In [1]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"


# 1. Read data

In [2]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/alleles/"

In [3]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-B*27:05', 'HLA-B*07:02', 'HLA-A*03:01', 'HLA-A*02:01', 'HLA-A*01:01']

## 2.0 Merge alleles together

In [4]:
# combination_size = len(ALLELES)
ALLELES = ['HLA-A*01:01', 'HLA-A*03:01']
combination_size = 2
TOTAL_SPLITS = 5

In [5]:
import itertools
per_allele_per_kfold_per_length_binders_train_new = dict()
per_allele_per_kfold_per_length_binders_test_new = dict()
NEW_ALELLES = list()
selected_lengths = np.arange(8,20)
for combination in itertools.combinations(ALLELES, combination_size):
    allele_name = "+".join(combination)
    NEW_ALELLES.append(allele_name)
    print(allele_name)
    per_allele_per_kfold_per_length_binders_train_new[allele_name] = dict()
    per_allele_per_kfold_per_length_binders_test_new[allele_name] = dict()

    for split_num in range(TOTAL_SPLITS):
        per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num] = dict()
        per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num] = dict()

        list_of_length_sets = list()
        for old_allele in combination:
            lengths = set(per_allele_per_kfold_per_length_binders_train[old_allele][split_num].keys())
            list_of_length_sets.append(lengths)
        intersected_lengths = set.intersection(*list_of_length_sets)

        for target_length in intersected_lengths:
            data_arrays_train = list()
            data_arrays_test = list()
            if split_num == 0:
                print(f"Length {target_length}:" , end=" ")
            for old_allele in combination:
                single_allele_data_array_train = per_allele_per_kfold_per_length_binders_train[old_allele][split_num][target_length]
                if split_num == 0:
                    print(f"{old_allele} - {len(single_allele_data_array_train)}", end= " ")
                data_arrays_train.append(single_allele_data_array_train)

                single_allele_data_array_test = per_allele_per_kfold_per_length_binders_test[old_allele][split_num][target_length]
                if split_num == 0:
                    # print(f"{old_allele} - {len(single_allele_data_array_test)}", end= " ")
                    pass
                data_arrays_test.append(single_allele_data_array_test)
            print()

            per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_train)
            per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_test)




per_allele_per_kfold_per_length_binders_train = per_allele_per_kfold_per_length_binders_train_new
per_allele_per_kfold_per_length_binders_test = per_allele_per_kfold_per_length_binders_test_new
ALLELES = NEW_ALELLES

HLA-A*01:01+HLA-A*03:01
Length 8: HLA-A*01:01 - 764 HLA-A*03:01 - 93 
Length 9: HLA-A*01:01 - 5052 HLA-A*03:01 - 5132 
Length 10: HLA-A*01:01 - 2464 HLA-A*03:01 - 2184 
Length 11: HLA-A*01:01 - 1430 HLA-A*03:01 - 691 
Length 12: HLA-A*01:01 - 1106 HLA-A*03:01 - 202 
Length 13: HLA-A*01:01 - 842 HLA-A*03:01 - 98 
Length 14: HLA-A*01:01 - 568 HLA-A*03:01 - 31 
Length 15: HLA-A*01:01 - 464 HLA-A*03:01 - 18 
Length 16: HLA-A*01:01 - 345 HLA-A*03:01 - 12 
Length 17: HLA-A*01:01 - 209 HLA-A*03:01 - 5 
Length 18: HLA-A*01:01 - 128 HLA-A*03:01 - 4 














































# 2. Train HMMS
## 2.1 Code for HMM
### 2.1.1 Training code

In [6]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions

TARGET_LENGTHS= [9]
NUM_RUNS = 2

## 2.2 Train
### 2.2.1 Train 8-mer
#### 2.2.1.1 baum-welch

In [7]:
stop_threshold = 0
cutoff = 0.001
per_symbol_portion = -22 / 9
for target_length in TARGET_LENGTHS:
    target_length_for_training = target_length
    train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length])
    ## Stop threshold Hardcoded here
    stop_threshold +=  abs(cutoff * train_size * per_symbol_portion  * target_length_for_training)
stop_threshold = stop_threshold / len(TARGET_LENGTHS)
stop_threshold

224.04800000000003

In [8]:
%%time
algorithm = 'viterbi'

per_name_models = dict()
per_allele_per_run_per_split_models=dict()
for allele_name in ALLELES:
    print(f"allele {allele_name}")
    per_allele_per_run_per_split_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        print(f"run {run_num}")

        print()


        per_kfold_per_length_data =  per_allele_per_kfold_per_length_binders_train[allele_name]

        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        verbose=True,
                                        multiple_check_input=False,
                                        algorithm=algorithm,
                                        name_prefix=f"{run_num}_run",
                                        num_states=27,
                                        n_jobs=10,
                                        stop_threshold=stop_threshold,
                                        min_iterations=5,
                                        emission_pseudocount=10
                                        )
        per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)


allele HLA-A*01:01+HLA-A*03:01
run 0

split 0
train size 10184
Shuffling for fairness 
Simple training: Train Length 10184, init_batch_size None
[1] Improvement: 27336.56675286917	Time (s): 1.353
[2] Improvement: 0.7608151893946342	Time (s): 1.454
[3] Improvement: 0.4578599684755318	Time (s): 1.349
[4] Improvement: 0.0	Time (s): 1.383
[5] Improvement: 0.0	Time (s): 1.448
Total Training Improvement: 27337.78542802704
Total Training Time (s): 9.0680
run 1

split 0
train size 10184
Shuffling for fairness 
Simple training: Train Length 10184, init_batch_size None
[1] Improvement: 23591.931812662864	Time (s): 1.396
[2] Improvement: 0.0	Time (s): 1.399
[3] Improvement: 0.0	Time (s): 1.405
[4] Improvement: 0.0	Time (s): 1.358
[5] Improvement: 0.0	Time (s): 1.452
Total Training Improvement: 23591.931812662864
Total Training Time (s): 8.6170
Wall time: 18 s


In [9]:
from hmm_visualization_methods import *

In [10]:
%%time

for i, model in enumerate(per_name_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
Wall time: 1min 2s


In [11]:
%%time

for i, model in enumerate(per_name_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
Wall time: 33.6 s


In [12]:
per_name_pyviz = dict()
for name, model in per_name_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-27-st-viterbi-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01-0) (Delete zero edges)
new starting point
Converting graph for model (1_run-9-27-st-viterbi-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01-0) (Delete zero edges)
new starting point


# Viterbi step

In [13]:
from hmm_logic_methods import copy_model_with_pseudocounts

In [14]:
per_allele_per_run_per_split_models[ALLELES[0]].keys()

dict_keys([0, 1])

In [33]:
per_allele_per_run_per_split_prepared_models = dict()
per_allele_per_run_per_split_new_histories = dict()

for allele_name in ALLELES:
    per_allele_per_run_per_split_prepared_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()

        for split_num, model in per_allele_per_run_per_split_models[allele_name][run_num].items():
            prepared_model = copy_model_with_pseudocounts(model, edge_pseudocount=10, precision=3)
            prepared_model.name = prepared_model.name + "_plus_viterbi"

            for state in prepared_model.states:
                if state is not prepared_model.start and state is not prepared_model.end:
                    state.distribution.thaw()
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

 N is  27
sum from start 1.0
 N is  27
sum from start 1.0


In [34]:
per_name_new_models = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train[allele_name]
        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        multiple_check_input=False,
                                        algorithm='viterbi',
                                        verbose=True,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        stop_threshold=stop_threshold,
                                        min_iterations= 5,
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num],
                                        emission_pseudocount=10,
                                        use_pseudocount=True)
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_new_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)

split 0
train size 10184
Shuffling for fairness 
Simple training: Train Length 10184, init_batch_size None
[1] Improvement: 41.98449184015044	Time (s): 1.127
[2] Improvement: -1.1833404570352286	Time (s): 1.222
[3] Improvement: 0.0	Time (s): 1.449
[4] Improvement: 0.0	Time (s): 1.212
[5] Improvement: 0.0	Time (s): 1.08
Total Training Improvement: 40.80115138311521
Total Training Time (s): 7.3970
split 0
train size 10184
Shuffling for fairness 
Simple training: Train Length 10184, init_batch_size None
[1] Improvement: 105.25857403560076	Time (s): 1.212
[2] Improvement: -5.631155513896374	Time (s): 1.171
[3] Improvement: -2.252973166090669	Time (s): 1.066
[4] Improvement: -1.423629433294991	Time (s): 1.029
[5] Improvement: 3.087579683342483	Time (s): 1.055
Total Training Improvement: 99.03839560566121
Total Training Time (s): 6.6980


In [35]:
per_name_new_models.keys()

dict_keys(['0_run-9-27-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01-0_plus_viterbi', '1_run-9-27-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01-0_plus_viterbi'])

In [36]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
Wall time: 596 ms


In [37]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
Wall time: 35.7 s


In [30]:
model = per_allele_per_run_per_split_models[allele_name][run_num][0]


In [26]:
model = per_allele_per_run_per_split_prepared_models[allele_name][run_num][0]

In [31]:
len(model.graph.edges())

783

In [32]:
for d in model.graph.edges(data=True):
    if d[0].name == d[1].name:
        print(d[2])

{'probability': -16.91982424654786, 'pseudocount': 0.037037037037037035, 'group': None}
{'probability': -19.386122389724292, 'pseudocount': 0.037037037037037035, 'group': None}
{'probability': -18.65346111673948, 'pseudocount': 0.037037037037037035, 'group': None}
{'probability': -8.410549965053423, 'pseudocount': 0.037037037037037035, 'group': None}
{'probability': -16.268352064163597, 'pseudocount': 0.037037037037037035, 'group': None}
{'probability': -24.642710924164415, 'pseudocount': 0.037037037037037035, 'group': None}
{'probability': -22.28724979324363, 'pseudocount': 0.037037037037037035, 'group': None}
{'probability': -20.826041720337038, 'pseudocount': 0.037037037037037035, 'group': None}
{'probability': -32.71248758070605, 'pseudocount': 0.037037037037037035, 'group': None}
{'probability': -20.334605195034158, 'pseudocount': 0.037037037037037035, 'group': None}
{'probability': -17.520287771546588, 'pseudocount': 0.037037037037037035, 'group': None}
{'probability': -26.522776

In [25]:
per_name_pyviz = dict()
for name, model in per_name_new_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-27-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01-0_plus_viterbi) (Delete zero edges)
Converting graph for model (1_run-9-27-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01-0_plus_viterbi) (Delete zero edges)


In [62]:
model = per_name_models['0_run-9-10-40-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0']

In [67]:
peptides = per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][10][:50]

In [69]:
for peptide in peptides:
    logp, path = model.viterbi(peptide)
    print("Viterbi for Sequence: '{}'  -- Log Probability: {} -- Path: {}".format(''.join( peptide ), logp, " ".join( state.name for idx, state in path[1:-1] ) ))

Viterbi for Sequence: 'LLFNILGGWV'  -- Log Probability: -30.897085708807133 -- Path: s031 s004 s017 s023 s010 s012 s021 s030 s035 s016
Viterbi for Sequence: 'TLYEHNNELV'  -- Log Probability: -30.174258369728715 -- Path: s011 s004 s017 s023 s010 s012 s002 s022 s026 s016
Viterbi for Sequence: 'DAVTYTEHAK'  -- Log Probability: -38.32822062561017 -- Path: s038 s000 s034 s028 s032 s019 s002 s022 s026 s006
Viterbi for Sequence: 'LLGVGNNLLV'  -- Log Probability: -28.79453839787274 -- Path: s031 s009 s025 s027 s015 s012 s021 s030 s035 s016
Viterbi for Sequence: 'YLIGGDVKSL'  -- Log Probability: -27.016003243469335 -- Path: s031 s004 s017 s028 s032 s018 s002 s007 s039 s016
Viterbi for Sequence: 'KLPEKLQAEV'  -- Log Probability: -27.749035748954768 -- Path: s011 s004 s017 s023 s010 s002 s022 s026 s024 s016
Viterbi for Sequence: 'TLRNTNPNFV'  -- Log Probability: -33.06197281224592 -- Path: s011 s004 s017 s023 s010 s012 s021 s030 s035 s016
Viterbi for Sequence: 'KGDALVVAQL'  -- Log Probability: -3

In [71]:
for peptide in peptides:
    logp, path = model.maximum_a_posteriori(list(peptide))
    print("Max a post for Sequence: '{}'  -- Log Probability: {} -- Path: {}".format(''.join( peptide ), logp, " ".join( state.name for idx, state in path ) ))

Max a post for Sequence: 'LLFNILGGWV'  -- Log Probability: -4.194813896394699 -- Path: s031 s004 s017 s023 s008 s012 s021 s030 s035 s016
Max a post for Sequence: 'TLYEHNNELV'  -- Log Probability: -4.687950610946164 -- Path: s011 s004 s017 s023 s010 s012 s002 s022 s035 s016
Max a post for Sequence: 'DAVTYTEHAK'  -- Log Probability: -5.915240768861771 -- Path: s038 s000 s034 s027 s032 s019 s002 s022 s026 s006
Max a post for Sequence: 'LLGVGNNLLV'  -- Log Probability: -4.9860346832952835 -- Path: s031 s009 s025 s027 s015 s012 s021 s030 s035 s016
Max a post for Sequence: 'YLIGGDVKSL'  -- Log Probability: -3.861137682186069 -- Path: s031 s004 s017 s023 s032 s018 s002 s007 s039 s016
Max a post for Sequence: 'KLPEKLQAEV'  -- Log Probability: -5.62211216923685 -- Path: s011 s004 s017 s023 s010 s002 s022 s026 s024 s016
Max a post for Sequence: 'TLRNTNPNFV'  -- Log Probability: -5.1468730944201155 -- Path: s011 s004 s017 s023 s010 s012 s021 s030 s035 s016
Max a post for Sequence: 'KGDALVVAQL'  -

In [70]:
for peptide in peptides:
    logp = model.log_probability(peptide)
    print(" Sequence: '{}'  -- Log Probability: {} ".format(''.join( peptide ), logp, ))

 Sequence: 'LLFNILGGWV'  -- Log Probability: -27.35635784453421 
 Sequence: 'TLYEHNNELV'  -- Log Probability: -26.669716509387314 
 Sequence: 'DAVTYTEHAK'  -- Log Probability: -34.58069294800763 
 Sequence: 'LLGVGNNLLV'  -- Log Probability: -25.4491279006268 
 Sequence: 'YLIGGDVKSL'  -- Log Probability: -23.739935954318444 
 Sequence: 'KLPEKLQAEV'  -- Log Probability: -24.3796213417221 
 Sequence: 'TLRNTNPNFV'  -- Log Probability: -28.981879313425626 
 Sequence: 'KGDALVVAQL'  -- Log Probability: -27.401631532592777 
 Sequence: 'ALVLLILMTA'  -- Log Probability: -26.5205643654398 
 Sequence: 'VLAALVCYIV'  -- Log Probability: -27.028198435962654 
 Sequence: 'FLAEEGTAKL'  -- Log Probability: -22.91004149657301 
 Sequence: 'YISQFSYKEL'  -- Log Probability: -29.18135332232505 
 Sequence: 'KLRRGDLPFV'  -- Log Probability: -26.546691292883548 
 Sequence: 'YLREADFARV'  -- Log Probability: -26.816801350301947 
 Sequence: 'LLTSFTSQAV'  -- Log Probability: -25.878879078189577 
 Sequence: 'LLMIIGAL

In [72]:
peptides = per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][9][:50]

In [73]:
for peptide in peptides:
    logp, path = model.viterbi(peptide)
    print("Viterbi for Sequence: '{}'  -- Log Probability: {} -- Path: {}".format(''.join( peptide ), logp, " ".join( state.name for idx, state in path[1:-1] ) ))

Viterbi for Sequence: 'TLQSKDFKL'  -- Log Probability: -24.843949307986314 -- Path: s011 s004 s037 s023 s018 s021 s030 s039 s016
Viterbi for Sequence: 'LLLEVSEEL'  -- Log Probability: -22.586736166927015 -- Path: s031 s004 s017 s023 s008 s014 s022 s026 s016
Viterbi for Sequence: 'VVVKILKPV'  -- Log Probability: -27.85772417437325 -- Path: s031 s004 s037 s023 s010 s002 s007 s024 s016
Viterbi for Sequence: 'AVMQEVPLA'  -- Log Probability: -27.498095487722733 -- Path: s011 s004 s037 s023 s018 s002 s022 s035 s006
Viterbi for Sequence: 'GMASALEAL'  -- Log Probability: -24.14621706592266 -- Path: s011 s004 s037 s023 s010 s002 s022 s026 s016
Viterbi for Sequence: 'LLSIIFFPA'  -- Log Probability: -28.217037818536053 -- Path: s031 s004 s017 s029 s019 s021 s030 s035 s006
Viterbi for Sequence: 'IDKFVPQMP'  -- Log Probability: -35.05227726365399 -- Path: s038 s000 s034 s027 s001 s020 s036 s035 s006
Viterbi for Sequence: 'YIQKTLFKI'  -- Log Probability: -25.55994793216949 -- Path: s031 s004 s037 s0

In [74]:
for peptide in peptides:
    logp, path = model.maximum_a_posteriori(list(peptide))
    print("Max a post for Sequence: '{}'  -- Log Probability: {} -- Path: {}".format(''.join( peptide ), logp, " ".join( state.name for idx, state in path ) ))

Max a post for Sequence: 'TLQSKDFKL'  -- Log Probability: -3.0215446387905267 -- Path: s011 s004 s037 s023 s018 s005 s030 s039 s016
Max a post for Sequence: 'LLLEVSEEL'  -- Log Probability: -3.9786279833205036 -- Path: s031 s004 s017 s023 s008 s014 s022 s026 s016
Max a post for Sequence: 'VVVKILKPV'  -- Log Probability: -4.532896953537236 -- Path: s031 s004 s037 s023 s010 s002 s007 s035 s016
Max a post for Sequence: 'AVMQEVPLA'  -- Log Probability: -4.267698412846126 -- Path: s011 s004 s037 s023 s010 s002 s022 s035 s006
Max a post for Sequence: 'GMASALEAL'  -- Log Probability: -3.629362203923499 -- Path: s011 s004 s037 s023 s010 s002 s022 s026 s016
Max a post for Sequence: 'LLSIIFFPA'  -- Log Probability: -6.718725243841966 -- Path: s031 s004 s017 s027 s001 s020 s030 s035 s006
Max a post for Sequence: 'IDKFVPQMP'  -- Log Probability: -3.6019261637640767 -- Path: s038 s000 s034 s027 s001 s020 s022 s035 s006
Max a post for Sequence: 'YIQKTLFKI'  -- Log Probability: -2.262097506748866 -- 